### ספריות


In [1]:
import pandas as pd
import os
import sys

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

### העלת משתנים להרצת הקוד


In [2]:
# מיקום תיקייה נוכחית
cwd = os.getcwd()

education_folder_path = os.path.dirname(cwd)

In [3]:
# תאריך
file_date=pd.Timestamp.today().strftime('%y%m%d')

### פונקציות גלובליות


In [4]:
# הוספת נתיב modules כנתיב יחסי
sys.path.append('../modules')

from global_functions import remove_spaces_in_columns, up_load_df

### העלאת טבלאות


In [5]:
# בתי ספר וגנים מעיריית בית שמש
BShemesh=up_load_df(r'{}\background_files\betshemesh_muni\חינוך'.format(education_folder_path),'מצבת מוסדות חינוך תשפג משרדי')
BShemesh=remove_spaces_in_columns(BShemesh)

### עיבוד


In [6]:
# מחיקת שורות של גני ילדים
BShemesh=BShemesh[~BShemesh['הערות'].str.contains('גן', na=False)]

In [7]:
BShemesh=BShemesh.drop(columns=['מגזר', "שכונה", 'טלפון','פקס', "מייל", "שם_מנהל/ת", "כתובת_מנהל", "שם_מזכיר/ה", "נייד.1", "הערות", 'מפקח', 'קב"ט_מוס"ח', 'גן_טרום_חובה', 'גן_חובה', 'גני_חינוך_מיוחד', 'סה"כ_גנים', 'כיתה_א1', 'כיתה_א2', 'כיתה_א3', 'כיתה_א4', 'כיתה_א5', 'כיתה_מקדמת',  'נייד',
 'כיתה_ב1', 'כיתה_ב2', 'כיתה_ב3', 'כיתה_ב4', 'כיתה_מקדמת.1', 'כיתה_ג1', 'כיתה_ג2', 'כיתה_ג3', 'כיתה_ג4', 'כיתה_מקדמת.2', 'כיתה_ד1', 'כיתה_ד2', 'כיתה_ד3', 'כיתה_ד4', 'כיתה_מקדמת.3', 'כיתה_ה1', 'כיתה_ה2', 'כיתה_ה3', 'כיתה_ה4', 'כיתה_מקדמת.4',
 'כיתה_ו1', 'כיתה_ו2', 'כיתה_ו3', 'כיתה_ו4', 'כיתה_מקדמת.5',
 'סה"כ_יסודי_מ"מ_וממ"ד', 'כיתה_ז1', 'כיתה_ז2', 'כיתה_ז3', 'כיתה_ז4', 'כיתה_מקדמת.6', 'כיתה_ח1', 'כיתה_ח2', 'כיתה_ח3', 'כיתה_ח4', 'כיתה_מקדמת.7',
 'סה"כ_יסודי_חרדי', 'כיתה_ט1', 'כיתה_ט2', 'כיתה_ט3', 'כיתה_מקדמת.8', 'כיתה_י1', 'כיתה_י2', 'כיתה_י3', 'כיתה_מקדמת.9', 'כיתה_יא1', 'כיתה_יא2', 'כיתה_יא3', 'כיתה_מקדמת.10', 'כיתה_יב1', 'כיתה_יב2', 'כיתה_יב3', 'כיתה_מקדמת.11', '_סה"כ_על_יסודי_מ"מ_וממ"ד', 'סה"כ_על_יסודי_חרדי'
])

In [8]:
BShemesh = BShemesh[~BShemesh['סמל_מוסד'].isna()]

In [9]:
# # מחיקת השורות הכפולות מהטבלה המקורית
BShemesh_without_duplicates = BShemesh.drop_duplicates(subset='סמל_מוסד')
len(BShemesh_without_duplicates)

167

In [10]:
# קאורדינטות של מוסדות חינוך במרחב ירושלים
BShemesh_moe_mosdot_coordinates=up_load_df(r'{}\Intermediates'.format(cwd),'250121_BShemesh_moe_mosdot_coordinates_2022')
BShemesh_moe_mosdot_coordinates=remove_spaces_in_columns(BShemesh_moe_mosdot_coordinates)
len(BShemesh_moe_mosdot_coordinates)

160

In [11]:
BShemeshNaN = BShemesh_without_duplicates[~BShemesh_without_duplicates['סמל_מוסד'].isin(BShemesh_moe_mosdot_coordinates['סמל_מוסד'])]
BShemeshNaN.to_excel(r'{}\Intermediates\{}_BShemeshNaN.xlsx'.format(cwd, file_date), index=False)


In [12]:
# מיזוג הטבלאות
BShemesh_moe_coordinates = pd.merge(
    BShemesh_without_duplicates,
    BShemesh_moe_mosdot_coordinates,
    on='סמל_מוסד',
    suffixes=('_BShemesh_without_duplicates', '_BShemesh_moe_mosdot_coordinates')  # מוסיף סיומות לשמות עמודות זהים
)

In [13]:
BShemesh_moe_coordinates=BShemesh_moe_coordinates.drop(columns=['שם_המוסד', 'כתובת_BShemesh_without_duplicates', 'יישוב', 'כתובת_BShemesh_moe_mosdot_coordinates'])

BShemesh_moe_coordinates.to_excel(r'{}\background_files\{}_BShemesh_moe_coordinates.xlsx'.format(education_folder_path, file_date), index=False)